# `NSE` json tests

In [39]:
# hack to change jupyter directory in notebooks for imports
import os
from pathlib import Path
if Path.cwd().parts[-1] == 'notebooks':
    root = Path.cwd().parent
else:
    root = Path.cwd()
os.chdir(root)

# Logger
import logging, logging.config
import yaml

with open(root / 'config' / 'log.yml', 'r') as f:
    config = yaml.safe_load(f.read())
    logging.config.dictConfig(config)

log = logging.getLogger('ib_log')

## Reading json files from NSE website

In [ ]:
from src.nse.nse import nse_json

import pandas as pd
import numpy as np

symbol = "INFY"

## Getting ALL equity prices

In [40]:
# testing equity price watch url

url = "https://www1.nseindia.com/live_market/dynaContent/live_watch/stock_watch/foSecStockWatch.json"

js = nse_json(url)
x = []

for item in js['data']:
    df = pd.DataFrame.from_dict([item])
    x.append(df)
df = pd.concat(x, ignore_index=True)

df.head()

,symbol,open,high,low,ltP,ptsC,per,trdVol,trdVolM,ntP,mVal,wkhi,wklo,wkhicm_adj,wklocm_adj,xDt,cAct,yPC,mPC
0,PVR,"1,504.95","1,535.75","1,494.80","1,530.00",32.95,2.20,4.29,0.43,65.20,0.65,"2,214.85","1,483.00",749.00,465.00,31-DEC-2999,-,-11.27,-9.29
1,ZEEL,187.20,193.90,187.10,192.30,4.00,2.12,65.11,6.51,124.73,1.25,308.70,176.55,402.40,251.80,31-DEC-2999,-,-25.81,-11.93
2,TRENT,"1,331.10","1,351.00","1,296.95","1,351.00",26.90,2.03,6.27,0.63,83.17,0.83,"1,551.70",982.85,"1,575.90",950.00,31-DEC-2999,-,15.76,3.41
3,MCDOWELL-N,757.15,770.55,756.10,768.80,14.40,1.91,20.25,2.03,154.75,1.55,951.80,712.00,"2,940.80","2,225.00",31-DEC-2999,-,-10.07,-1.15
4,M&MFIN,237.00,244.70,237.00,242.90,4.40,1.84,35.24,3.52,85.09,0.85,272.00,142.40,356.50,229.25,31-DEC-2999,-,65.13,-6.85


## Getting ALL Index prices

In [65]:
# testing an index price watch url

url = 'https://iislliveblob.niftyindices.com/jsonfiles/LiveIndicesWatch.json'
js = nse_json(url) # works!

x = []
for item in js['data']:
    df = pd.DataFrame.from_dict([item])
    x.append(df)
df = pd.concat(x, ignore_index=True)

# keep only equities
df = df[(df.indexType == 'eq') & (df.yearHigh != '-')].reset_index(drop=True)

# symbol map
di = {'NIFTY 50': 'NIFTY50', 'NIFTY BANK': 'BANKNIFTY', 
      'INDIA VIX': 'INDIAVIX', 'NIFTY IT': 'NIFTYIT'}
df.insert(0, 'symbol', df.indexName.map(di).fillna(df.indexName))

important = True

# keep only important indices
if important:
   df = df[df.symbol.isin(di.values())]


In [66]:
df

,symbol,yearLow,last,indexName,yearHigh,previousClose,high,indexType,low,timeVal,percChange,indexOrder,open,indexSubType
0,NIFTY50,"15,183.40","17,043.30",NIFTY 50,"18,887.60","17,154.30","17,224.65",eq,"16,987.10",2023-03-14 15:33:07+05:30,-0.65,0.00,"17,160.55",bm
2,NIFTYIT,"26,186.70","28,784.70",NIFTY IT,"36,813.10","29,267.00","29,402.25",eq,"28,638.90",2023-03-14 15:33:07+05:30,-1.65,2.00,"29,281.25",sc
3,BANKNIFTY,"32,290.55","39,411.40",NIFTY BANK,"44,151.80","39,564.70","39,768.50",eq,"39,132.60",2023-03-14 15:33:07+05:30,-0.39,3.00,"39,522.40",sc
4,INDIAVIX,10.1725,16.2175,INDIA VIX,28.1300,16.2150,16.6425,eq,15.0025,2023-03-14 15:33:07+05:30,0.02,4.0000,16.2150,bm


## Checking if NSE is open now

In [ ]:
# testing a plan marketStatus url
url = 'https://nseindia.com/api/marketStatus'
js = nse_json(url) # works!
nse_is_open = js[list(js.keys())[0]][0]['marketStatus'] != 'Closed'
nse_is_open

## Parsing a csv file from NSE website
... can be used to liberate get_nse_syms!

In [ ]:
# testing a url with a csv file
# similar to src/nse/nse.py/get_nse_syms()

symbol = "" # Empty means all symbols

url = "https://archives.nseindia.com/content/fo/fo_mktlots.csv"
js = nse_json(url)

df = pd.read_csv(url)
df.columns = df.columns.str.strip().str.lower() # cleanup column names

# strip all string contents of whitespaces
df = df.applymap(lambda x: x.strip()
         if type(x) is str else x)

# remove 'Symbol' row
df = df[df.symbol != "Symbol"]

# introduce `secType`
df.insert(1, 'secType', np.where(df.symbol.str.contains('NIFTY'), 'IND', 'STK'))

# introduce `exchange`
df.insert(2, 'exchange', 'NSE')

if symbol:
    result = df[(df.iloc[:, 1] == symbol.upper())]
else:
    result = df


# Testing to liberate `history`    
...from nsepy and other dependancies


In [ ]:
# testing equity history
from functools import partial
from urllib.parse import urlparse


In [ ]:
url1 = "http://www1.nseindia.com/products/dynaContent/common/productsSymbolMapping.jsp"
partial(urlparse(url1))

In [ ]:


url2 = "symbol=SBIN&segmentLink=3&symbolCount=1&series=EQ&dateRange=1month&fromDate='01-01-2017'&toDate='01-01-2017'&dataType=PRICEVOLUMEDELIVERABLE"
partial(url1, url2)

In [ ]:

import requests

class URLFetch:

    def __init__(self, url, method='get', json=False, session=None,
                 headers = None, proxy = None):
        self.url = url
        self.method = method
        self.json = json

        if not session:
            self.session = requests.Session()
        else:
            self.session = session

        if headers:
            self.session.headers.update(headers)
        if proxy:
            self.update_proxy(proxy)
        else:
            self.update_proxy('')

    def set_session(self, session):
        self.session = session
        return self

    def get_session(self, session):
        self.session = session
        return self

    def __call__(self, *args, **kwargs):
        u = urlparse(self.url)
        self.session.headers.update({'Host': u.hostname})
        url = self.url%(args)
        if self.method == 'get':
            return self.session.get(url, params=kwargs, proxies = self.proxy )
        elif self.method == 'post':
            if self.json:
                return self.session.post(url, json=kwargs, proxies = self.proxy )
            else:
                return self.session.post(url, data=kwargs, proxies = self.proxy )

In [ ]:
eq_hist_partial = "http://www1.nseindia.com/products/dynaContent/common/productsSymbolMapping.jsp"

symbol="SBIN"
symbolCount=1
series="EQ"
fromDate="dd-mm-yyyy"
toDate="dd-mm-yyyy"
dd = "symbol='SBIN', series='EQ', fromDate='01-01-2017', toDate='01-01-2017'"